# TieredKV Phase 4 — audited 3% / 5% / 10% threshold experiment

This notebook is a runner, **not a completed benchmark**. Select a GPU runtime.
Use the team's existing trained 3B dictionary and its training JSON; this notebook does not substitute a random dictionary or retrain one silently.

All three thresholds use the same pinned model snapshot, dictionary, fixed GSM8K subset, 5-shot examples, generation settings, W=32, C=8, and strict 20% retained-KV byte policy. The strict policy adapts integer atom counts to actual CSR storage; it is separate from the fixed-uniform baseline policy.

The notebook checks real-model generation equality first. It only reports Phase 4 complete when all three accuracy runs pass the per-problem and aggregate memory audits. Negative accuracy results and failed audits remain saved.

**Edit DICT_SOURCE below.** The adjacent JSON must name the same MODEL_ID. N=20 is a small experiment; use the same larger N for all thresholds when compute permits. Results alone do not guarantee statistical significance.

In [ ]:
from pathlib import Path
import os, subprocess, sys

BRANCH = "b-allocator-phase4-final"
PROJECT = Path(os.environ.get("TIEREDKV_PROJECT","/content/KV-cache-phase4")).resolve()
if not PROJECT.exists():
    subprocess.run(["git","clone","--branch",BRANCH,
                    "https://github.com/samarth2006-28/KV-cache.git",str(PROJECT)],check=True)
else:
    current = subprocess.check_output(["git","branch","--show-current"],cwd=PROJECT,text=True).strip()
    dirty = subprocess.check_output(["git","status","--porcelain"],cwd=PROJECT,text=True).strip()
    if current != BRANCH or dirty:
        raise RuntimeError("Existing checkout is another branch or has edits. Choose a new PROJECT path.")
    subprocess.run(["git","pull","--ff-only"],cwd=PROJECT,check=True)
os.chdir(PROJECT)
# Notebook kernels can retain an absolute sys.path after changing folders.
if str(PROJECT) not in sys.path:
    sys.path.insert(0,str(PROJECT))
import importlib
importlib.invalidate_caches()
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"],check=True)
print("Code commit:",subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip())

In [ ]:
import torch
from importlib.metadata import version
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory/2**30,1), "GiB")
print("Torch",torch.__version__,"Transformers",version("transformers"))
if version("transformers") != "4.49.0":
    raise RuntimeError("Setup did not install Transformers 4.49.0. Re-run the setup cell.")
if props.total_memory < 10*2**30:
    raise RuntimeError("This 3B workflow requires at least 10 GiB VRAM for weights and inference buffers.")

## Assets and experiment settings

Mount your Drive and point DICT_SOURCE to the team's trained dictionary. The training metadata must sit beside it with the same basename and a .json suffix. Default model ID matches the committed 3B training metadata.

If you know the model commit used for training, enter MODEL_REVISION. Otherwise the next cell resolves and records one exact commit before downloading. Existing training metadata without a model commit cannot prove exact training-revision identity; that limitation stays visible.

For a gated model, authenticate in your own runtime using huggingface_hub.notebook_login(). Never paste a token into notebook source.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"
MODEL_REVISION = ""  # exact training model commit, if recorded
DICT_SOURCE = Path("/content/drive/MyDrive/tieredkv/dicts/llama32_3b_N4096_s16.pt")
EXPECTED_DICT_SHA256 = ""  # optional trusted hash from the dictionary owner
N = 20
SEED = 0
MAX_NEW_TOKENS = 256
SAVE_ROOT = Path("/content/drive/MyDrive/tieredkv_phase4_runs")

In [ ]:
import json, shutil
from huggingface_hub import HfApi, snapshot_download
from tieredkv.notebook_support import dictionary_metadata

metadata = dictionary_metadata(DICT_SOURCE,MODEL_ID,EXPECTED_DICT_SHA256 or None)
training_revision = metadata["training"].get("model_revision") or metadata["training"].get("revision")
if MODEL_REVISION and training_revision and MODEL_REVISION != training_revision:
    raise ValueError("Requested model revision differs from dictionary training metadata.")
REVISION = MODEL_REVISION or training_revision or HfApi().model_info(MODEL_ID).sha
if len(REVISION) != 40:
    raise ValueError("Use an exact model commit SHA.")
if not training_revision:
    print("Note: dictionary training metadata did not record an exact model commit.")
print("Model:",MODEL_ID,"commit:",REVISION)
print("Dictionary SHA256:",metadata["sha256"])

ASSETS = Path("/content/phase4_assets")
ASSETS.mkdir(exist_ok=True)
DICT = ASSETS/"shared_dictionary.pt"
shutil.copy2(DICT_SOURCE,DICT)
shutil.copy2(DICT_SOURCE.with_suffix(".json"),DICT.with_suffix(".json"))
MODEL = snapshot_download(repo_id=MODEL_ID,revision=REVISION,
    ignore_patterns=["*.bin","*.gguf","*.h5","*.ot","original/*"])
print("Pinned snapshot ready:",MODEL)

## Optional offline regression checks

These check code compatibility. The experiment runner separately runs five actual target-model generation gates, including the real encoder, before starting accuracy measurements.

In [ ]:
os.environ.pop("TIEREDKV_MODEL",None)
subprocess.run([sys.executable,"-m","pytest","-q","tests/test_allocator.py",
                "tests/test_cache_compress.py","tests/test_phase4_budget.py",
                "tests/test_phase4_experiment.py"],check=True,cwd=str(PROJECT))

## Run all three thresholds

Results go directly to Drive. Each run has unique predictions and manifests. The final bundle includes independent answer rescoring, actual retained-KV audits, Wilson intervals and paired McNemar tests with three-comparison Bonferroni correction.

If the runner fails or a budget is invalid, inspect experiment.json. Do not change the question subset after seeing results, relabel memory, or present failed cells as valid comparisons. Full prefill still materializes full KV and eager attention; this is not a peak-memory optimization.

The run streams progress and saves a .log on Drive. A failed child process raises a visible error with its final traceback. After a failure, you can still run the result/download cells to inspect saved artifacts. When updating this notebook, reopen the GitHub version: an older Drive copy does not update automatically.

In [ ]:
from datetime import datetime, timezone
import uuid
SAVE_ROOT.mkdir(parents=True,exist_ok=True)
OUT = SAVE_ROOT/("phase4_"+datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:8])
command = [sys.executable,"-u","-m","tieredkv.phase4_experiment",
           "--model",MODEL,"--model-id",MODEL_ID,"--model-revision",REVISION,
           "--dictionary",str(DICT),"--out",str(OUT),"--n",str(N),
           "--seed",str(SEED),"--max-new-tokens",str(MAX_NEW_TOKENS),
           "--dictionary-sha256",metadata["sha256"]]
from tieredkv.notebook_support import run_logged
LOG = Path(str(OUT)+".log")
print("Starting Phase 4. Live output below; full log:",LOG,flush=True)
run_logged(command,LOG,result_dir=OUT,cwd=str(PROJECT))
print("Results:",OUT)
print("Archive:",str(OUT)+".zip")

In [ ]:
report_path = OUT/"experiment.json"
if not report_path.exists():
    raise RuntimeError("Preflight failed before result creation; inspect the preceding error.")
report = json.loads(report_path.read_text())
print("Status:",report["status"])
print("Phase 4 completed:",report["phase4_completed"])
for method,cell in report.get("cells",{}).items():
    print(method, f'{cell["strict_correct"]}/{cell["n"]} strict correct',
          f'mean KV={100*cell["mean_budget"]:.3f}%',
          "eligible=",cell["eligible"],"budget failures=",cell["budget_failures"])
for pair in report.get("paired_mcnemar",[]):
    print(pair["a"],"vs",pair["b"],"adjusted p=",pair["p_bonferroni"])
if not report["phase4_completed"]:
    print("No completed threshold comparison. Error:",report.get("error","see budget audits"))
else:
    from IPython.display import display, Markdown, Image
    display(Markdown((OUT/"table.md").read_text()))
    display(Image(filename=str(OUT/"ablation.png")))
    print("Report observed differences and uncertainty; do not infer superiority from a tiny sample.")

In [ ]:
from google.colab import files
archive = Path(str(OUT)+".zip")
if archive.is_file():
    files.download(str(archive))
else:
    print("No archive yet. The experiment directory and progress files remain on Drive.")